[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/magrilu/cv-dojo/blob/main/notebooks/two-views/eight-point.ipynb)

# The fundamental matrix

This notebook explores epipolar geometry, the geometric and algebraic constraint that ties two uncalibrated views of the same 3D scene.

We start from a question. Given a point $\mathbf{x}$ in the first image and a
point $\mathbf{x}'$ in the second, how do we decide whether they are images of
the same 3D point $\mathbf{X}$?

This can be answered from the image coordinates alone, without relying on image content or the reconstruction of 3D point. Specifically, there is a condition that involves only the two image points and
the two cameras that acquired the scene and it reads

$$\mathbf{x}'^\top \mathsf{F}\, \mathbf{x} = 0 .$$

We first characterise corresponding points algebraically, and find that their
images must satisfy the previous bilinear relation. We then revisit that condition
geometrically to introduce one of the main characters of epipolar geometry, the fundamental
matrix. Finally we estimate it from correspondences on real images using the eight-point algorithm.

As usual the running examples are built around the Origami House. We will work the
geometry out in a noiseless setting before moving to real data.

In [ ]:
#| echo: false
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules and not Path("cv-dojo").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/magrilu/cv-dojo.git"], check=True)
    import os
    os.chdir("cv-dojo")

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "cvdojo").exists():
        sys.path.insert(0, str(parent / "src")); break

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection

from cvdojo.house import (load_image, load_model, load_two_view_cameras,
                          load_annotation, project)
from cvdojo.plotting import (clip_line_to_image, draw_line_in_image,
                             pairwise_intersections, ACCENT)
from cvdojo.scene import skew, set_axes_equal

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": False})

BLUE, GREY = "#3288BD", "0.45"

model = load_model()
V3 = {k: np.array(v, float) for k, v in model["vertices"].items()}
EDGES = model["edges"]
P, Pp = load_two_view_cameras()

I  = load_image("IMG_4331.jpeg")
Ip = load_image("IMG_4337.jpeg")
H_IMG, W_IMG = I.shape[:2]

IDS = list(V3)
X3 = np.array([V3[k] for k in IDS])
xs  = project(P,  X3)          # exact projections: our synthetic data
xps = project(Pp, X3)
h = lambda a: np.c_[np.atleast_2d(a), np.ones(len(np.atleast_2d(a)))]

In [ ]:
#| echo: false
#| column: page
#| label: fig-two-photographs
#| fig-cap: >-
#|   The same origami house from two views. Are $\mathbf{x}_1$ and $\mathbf{x}_1'$ a pair of corresponding points?
_ann0 = load_annotation("two_view_matches")
_m = next(m for m in _ann0["matches"] if m["id"] == "X1")

fig, axes = plt.subplots(1, 2, figsize=(12, 7.5), layout="constrained")
for ax, im, p, lab, ttl in [
        (axes[0], I,  _m["x"],  r"$\mathbf{x}_1$",  "view 1"),
        (axes[1], Ip, _m["xp"], r"$\mathbf{x}'_1$", "view 2")]:
    ax.imshow(im)
    ax.scatter(*p, s=140, facecolors="none", edgecolors=ACCENT, lw=2.4, zorder=5)
    ax.text(p[0] + 34, p[1] - 34, lab, color=ACCENT, fontsize=15,
            bbox=dict(fc="white", alpha=0.75, ec="none", pad=2))
    ax.set_xlim(150, 1250); ax.set_ylim(1750, 700)
    ax.set_title(ttl, fontsize=10); ax.axis("off")
fig.suptitle("Are those corresponding points?", fontsize=11)
plt.show()


## When do two points correspond?

Here in @ffig-object-and-views are the two views, drawn by projecting the model of the Origami House
through the two cameras $\mathsf{P}$ and $\mathsf{P}'$. For now we are working with synthetic images: every image point is projected exactly through the cameras.

Ten vertices are marked. The thin wireframe is there so that you can recognise
the house and see which corner is which.


In [ ]:
#| echo: false
#| column: page
#| label: fig-object-and-views
#| fig-cap: >-
#|   The ideal house in space, and its two exact perspective projections. 
fig = plt.figure(figsize=(16, 5.4), layout="constrained")

# --- the object itself, in three dimensions -------------------------------
ax0 = fig.add_subplot(1, 3, 1, projection="3d")
for a, b in EDGES:
    ax0.plot(*zip(V3[a], V3[b]), color=GREY, lw=1.0)
ctr = np.mean(list(V3.values()), axis=0)
for k, Q in V3.items():
    ax0.scatter(*Q, s=18, color=ACCENT, depthshade=False, zorder=5)
    off = Q - ctr
    off[:2] = 0.55 * off[:2] / (np.linalg.norm(off[:2]) or 1)
    off[2] = 0.30 * np.sign(off[2])
    ax0.text(*(Q + off), rf"$\mathbf{{X}}_{{{k[1:]}}}$",
             color=ACCENT, fontsize=9, ha="center", va="center")
d = model["dimensions_cm"]
ax0.set_box_aspect((d["long_side"], d["depth"], d["total_height"]))
ax0.view_init(elev=18, azim=-62); ax0.set_axis_off()
ax0.set_title("the object\n$\\mathbf{X}_i \\in \\mathbb{P}^3$", fontsize=11)

# --- and its two images ---------------------------------------------------
for j, (pts, ttl, prime) in enumerate(
        [(xs, r"view 1:  $\mathbf{x}_i = \mathsf{P}\,\mathbf{X}_i$", False),
         (xps, r"view 2:  $\mathbf{x}'_i = \mathsf{P}'\mathbf{X}_i$", True)]):
    ax = fig.add_subplot(1, 3, j + 2)
    Q = {k: p for k, p in zip(IDS, pts)}
    for a, b in EDGES:
        ax.plot(*zip(Q[a], Q[b]), color=GREY, lw=0.8)
    ax.scatter(pts[:, 0], pts[:, 1], s=40, facecolors="none",
               edgecolors=ACCENT, lw=1.6, zorder=5)
    for k, p in Q.items():
        lab = (rf"$\mathbf{{x}}'_{{{k[1:]}}}$" if prime
               else rf"$\mathbf{{x}}_{{{k[1:]}}}$")
        ax.text(p[0] + 16, p[1] - 16, lab, color=BLUE, fontsize=9)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_title(ttl, fontsize=11)
    ax.set_xlabel("u  [px]"); ax.set_ylabel("v  [px]")
plt.show()


Now let's start by the definition of corresponding points.

::: {#def-correspondence}
## Corresponding points

Two image points $\mathbf{x}$ and $\mathbf{x}'$ are **corresponding points** if there exists a world point $\mathbf{X}$ that projects to both:

$$\lambda\,\mathbf{x} = \mathsf{P}\mathbf{X}, \qquad
  \lambda'\mathbf{x}' = \mathsf{P}'\mathbf{X}.$$
:::

The scale factors $\lambda, \lambda'$ are there because a projective point is
only defined up to scale.

We can read the two equations as a single linear system in the unknowns
$(\mathbf{X}, \lambda, \lambda')$,  six unknowns, since $\mathbf{X}$ is a
homogeneous 4-vector:

$$
\underbrace{\begin{bmatrix}
\mathsf{P} & -\mathbf{x} & \mathbf{0} \\[2pt]
\mathsf{P}' & \mathbf{0} & -\mathbf{x}'
\end{bmatrix}}_{\textstyle \mathsf{L}}
\begin{bmatrix} \mathbf{X} \\ \lambda \\ \lambda' \end{bmatrix} = \mathbf{0}.
$$

$\mathsf{L}$ is $6 \times 6$ and the system is homogeneous, so it always has the
solution $\mathbf{0}$, which is not a valid solution at all, because $\mathbf{X} =
\mathbf{0}$ is not a point of $\mathbb{P}^3$. A genuine world point exists
exactly when the system has a **non-trivial** solution, that is, $\mathbf{x}$ and $\mathbf{x}'$ are corresponding points  if

$$\det \mathsf{L} = 0$$ {#eq-detL}


Let us observe what happens, when, for example, we substitute $\mathbf{x}_9$ and $\mathbf{x}'_9$ in @eq-detL:


In [ ]:
def L_matrix(P, Pp, x, xp):
    """The 6x6 system expressing that x and x' are images of a common point."""
    L = np.zeros((6, 6))
    L[:3, :4] = P;   L[:3, 4] = -x
    L[3:, :4] = Pp;  L[3:, 5] = -xp
    return L

k = "X9"                                  # the right end of the ridge
x  = h(xs[IDS.index(k)])[0]
xp = h(xps[IDS.index(k)])[0]

L = L_matrix(P, Pp, x, xp)
print(f"rank of L : {np.linalg.matrix_rank(L)}  (of 6)")
print(f"det L     : {np.linalg.det(L):.3e}")


The rank of $\mathsf L$ is five, and its determinant zero: the null space is one-dimensional, and it holds the homogeneous coordinates of a
single world point. Let us extract it and see what it is.


In [ ]:
sol = np.linalg.svd(L)[2][-1]
X_rec, lam, lamp = sol[:4], sol[4], sol[5]
X_rec = X_rec[:3] / X_rec[3]

print("recovered X :", np.round(X_rec, 3), "cm")
print("model    X  :", np.round(V3[k], 3), "cm")
print("difference  :", f"{np.linalg.norm(X_rec - V3[k]):.2e} cm")


So, as expected, $\mathbf{X}_9$ is indeed the solution of the system.

Now, you can carry out the same computation with a pair of points that are **not** corresponding points. We keep
$\mathbf{x}$ where it is, and take for $\mathbf{x}'$ the image of a different
vertex, the *left* end of the ridge instead of the right one.


In [ ]:
xp_wrong = h(xps[IDS.index("X8")])[0]        # image of a different vertex

L_bad = L_matrix(P, Pp, x, xp_wrong)
print(f"rank of L : {np.linalg.matrix_rank(L_bad)}  (of 6)")
print(f"det L     : {np.linalg.det(L_bad):.3e}")
print(f"ratio of the smallest to the largest singular value: "
      f"{np.linalg.svd(L_bad)[1][-1] / np.linalg.svd(L_bad)[1][0]:.2e}")


This time we get full rank. The only solution is the zero vector, which is not a
citizen of $\mathbb{P}^3$: the system has no valid solution: provided the given cameras, the two image points cannot be images of the same
corner of the house.

So we have a test: if two points correspond, then [-@eq-detL] holds.
Note that $\det\mathsf{L}$ is a number computed from four things — the two cameras $\mathsf P, \mathsf P'$ and the
two image points $\mathbf{x},\mathbf{x'}$. It does not involve the 3D point $\mathbf{X}$. The constraint has a nice property: $\mathbf{x}$ appears
in one column and nowhere else, and $\mathbf{x}'$ in another. A determinant is
linear in each of its columns. Therefore $\det\mathsf{L}$ is linear in
$\mathbf{x}$ and linear in $\mathbf{x}'$. In other words, it's a **bilinear form**. Every
bilinear form on $\mathbb{R}^3 \times \mathbb{R}^3$ can be written as

$$\det \mathsf{L} \;=\; \mathbf{x}'^\top \mathsf{F}\, \mathbf{x}$$

for a single $3 \times 3$ matrix $\mathsf{F}$. Since we are working in homogeneous coordinates, $\mathsf F$ is naturally defined up to a scalar factor. 

## The epipolar geometry


Let us look at the definition of corresponding points from a geometric point of view. The point
$\mathbf{x}$ in the first image  determines
the **optical ray**, the set of all world points that project there, running from the
camera centre $\mathsf{C}$ out through the image plane. The same is true of
$\mathbf{x}'$ and its ray from $\mathsf{C}'$.

Two points correspond exactly when their two rays meet.


In [ ]:
#| echo: false
#| column: page
#| label: fig-two-rays
#| fig-cap: >-
#|   Two optical rays that meet. A pair of image points corresponds exactly when
#|   the rays intersect somewhere in space.
def ray(P, x, tmin=-0.4, tmax=1.9, n=2):
    """Points along the optical ray of the image point x."""
    C = np.linalg.svd(P)[2][-1]; C = C[:3] / C[3]
    d = np.linalg.pinv(P) @ x; d = d[:3] / d[3] - C
    ts = np.linspace(tmin, tmax, n)
    return C, np.array([C + t * d for t in ts])

C  = np.linalg.svd(P)[2][-1];  C  = C[:3] / C[3]
Cp = np.linalg.svd(Pp)[2][-1]; Cp = Cp[:3] / Cp[3]
Xk = V3[k]

fig = plt.figure(figsize=(9, 6.5), layout="constrained")
ax = fig.add_subplot(111, projection="3d")
for a, b in EDGES:
    ax.plot(*zip(V3[a], V3[b]), color=GREY, lw=1.0)
for Cc, lab in [(C, r"$\mathsf{C}$"), (Cp, r"$\mathsf{C}'$")]:
    ax.scatter(*Cc, s=45, color=BLUE)
    ax.text(*(Cc + np.array([0.3, 0.3, 0.3])), lab, color=BLUE, fontsize=12)
    ax.plot(*zip(Cc, Xk), color=ACCENT, lw=1.4)
ax.scatter(*Xk, s=60, color=ACCENT)
ax.text(*(Xk + np.array([0.2, 0.2, 0.4])), r"$\mathbf{X}$", color=ACCENT, fontsize=12)
ax.set_axis_off(); set_axes_equal(ax); ax.view_init(elev=16, azim=-64)
ax.set_title("the two optical rays meet at the world point")
plt.show()


Three points are now in play: the two camera centres and the world point. Some names:

::: {#def-epipolar-plane}
## Epipolar plane and baseline

The plane through $\mathsf{C}$, $\mathsf{C}'$ and $\mathbf{X}$ is the
**epipolar plane** of $\mathbf{X}$.

The line $\mathsf{C}\mathsf{C}'$ is the **baseline**. It is the same for every
world point, so every epipolar plane contains it: they form a pencil of planes
hinged on the baseline.
:::

::: {#def-epipoles}
## Epipoles

The baseline meets the first image plane in a point $\mathbf{e}$ and the second
in $\mathbf{e}'$: the **epipoles**. Equivalently, $\mathbf{e}$ is the image of
the second camera centre in the first view, and $\mathbf{e}'$ the image of the
first centre in the second view.
:::

::: {#def-epipolar-line}
## Epipolar lines

The epipolar plane cuts each image plane in a line: the **epipolar lines**
$\boldsymbol{\ell}$ and $\boldsymbol{\ell}'$. Every epipolar line passes through
the epipole of its image, since the baseline lies in every epipolar plane.
:::


In [ ]:
#| echo: false
#| column: page
#| label: fig-epipolar-geometry
#| fig-cap: >-
#|   The whole vocabulary of epipolar geometry. The **baseline** joins the two
#|   centres and intersects each image plane at its **epipole**, $\mathbf e$ and
#|   $\mathbf e'$. The world point $\mathbf X$, together with the two centres,
#|   spans the **epipolar plane** (shaded), and that plane cuts each image in
#|   an **epipolar line** (depicted in red), which therefore passes through the epipole.
#|   The darker rectangle is the sensor, the paler one the image plane
#|   extended beyond it.
def look_at(Cc, target, f=900.0, up=np.array([0., 0., -1.])):
    z = target - Cc; z /= np.linalg.norm(z)
    x = np.cross(up, z); x /= np.linalg.norm(x)
    K = np.array([[f, 0, W_IMG/2], [0, f, H_IMG/2], [0, 0, 1.]])
    R = np.vstack([x, np.cross(z, x), z])
    return K @ np.hstack([R, (-R @ Cc).reshape(3, 1)])

CTR = np.mean(list(V3.values()), axis=0)
orbit = lambda az, d=26., el=np.deg2rad(30): CTR + d*np.array(
    [np.cos(el)*np.cos(np.deg2rad(az)), np.cos(el)*np.sin(np.deg2rad(az)), np.sin(el)])

Cd, Cpd = orbit(-118), orbit(-56)
Pd, Ppd = look_at(Cd, CTR), look_at(Cpd, CTR)
Xd = V3["X8"]                                   # the world point: a ridge vertex

def plane_map(Pm, Cc, depth):
    """Pixel coordinates -> points of the image plane, drawn `depth` ahead of Cc."""
    Mi = np.linalg.inv(Pm[:, :3])
    s = depth / np.linalg.norm(Mi @ np.array([W_IMG/2, H_IMG/2, 1.]))
    return lambda uv: Cc + s * (Mi @ np.array([uv[0], uv[1], 1.]))

def quad(f, corners): return np.array([f(c) for c in corners])
def spanning(pts, pad=.10):
    """A rectangle of pixel space holding the sensor and the given points."""
    A = np.vstack([[[0, 0], [W_IMG, 0], [W_IMG, H_IMG], [0, H_IMG]], np.atleast_2d(pts)])
    lo, hi = A.min(0), A.max(0); m = pad*(hi - lo); lo, hi = lo - m, hi + m
    return [(lo[0], lo[1]), (hi[0], lo[1]), (hi[0], hi[1]), (lo[0], hi[1])]

DEPTH = 3.0
g1, g2 = plane_map(Pd, Cd, DEPTH), plane_map(Ppd, Cpd, DEPTH)

# the epipoles straight from the definition: each centre seen by the other camera
ed  = Pd  @ np.append(Cpd, 1.); ed  = ed[:2]/ed[2]
epd = Ppd @ np.append(Cd,  1.); epd = epd[:2]/epd[2]
xd, xpd = project(Pd, Xd)[0], project(Ppd, Xd)[0]

SENSOR = [(0, 0), (W_IMG, 0), (W_IMG, H_IMG), (0, H_IMG)]
PL1, PL2 = quad(g1, spanning([ed, xd])), quad(g2, spanning([epd, xpd]))
SE1, SE2 = quad(g1, SENSOR), quad(g2, SENSOR)
E1, E2, Y1, Y2 = g1(ed), g2(epd), g1(xd), g2(xpd)

RED = "#C0392B"
fig = plt.figure(figsize=(13, 6.4))
ax = fig.add_subplot(111, projection="3d")

# the epipolar plane, as a patch of the plane through C, C' and X
nrm = np.cross(Cpd - Cd, Xd - Cd); nrm /= np.linalg.norm(nrm)
u = (Cpd - Cd)/np.linalg.norm(Cpd - Cd); v = np.cross(nrm, u)
S = np.vstack([Cd, Cpd, Xd, E1, E2, Y1, Y2])
ab = np.c_[(S - Cd) @ u, (S - Cd) @ v]
lo, hi = ab.min(0) - 1.5, ab.max(0) + 1.5
poly = np.array([Cd + a*u + b*v for a, b in
                 [(lo[0], lo[1]), (hi[0], lo[1]), (hi[0], hi[1]), (lo[0], hi[1])]])
ax.add_collection3d(Poly3DCollection([poly], alpha=.17, facecolor=ACCENT, edgecolor="none"))

for Q in (PL1, PL2):                       # the image plane, beyond the sensor
    ax.add_collection3d(Poly3DCollection([Q], alpha=.08, facecolor=BLUE,
                                         edgecolor=BLUE, lw=1.0))
for Q in (SE1, SE2):                       # the sensor itself
    ax.add_collection3d(Poly3DCollection([Q], alpha=.18, facecolor=BLUE,
                                         edgecolor=BLUE, lw=1.8))

for a, b in EDGES:
    ax.plot(*zip(V3[a], V3[b]), color=GREY, lw=1.6)

d = Cpd - Cd
ax.plot(*zip(Cd - .12*d, Cpd + .12*d), color=BLUE, lw=2.0)          # baseline
for Cc in (Cd, Cpd):                                                # optical rays
    ax.plot(*zip(Cc, Xd + .10*(Xd - Cc)), color="k", lw=1.5)
for A, B in ((Y1, E1), (Y2, E2)):                                   # epipolar lines
    ax.plot(*zip(A, B), color=RED, lw=3.2)

for Q, lab, col, off in ((Cd,  r"$\mathsf{C}$",   BLUE, [0, 0, 1.9]),
                         (Cpd, r"$\mathsf{C}'$",  BLUE, [0, 0, 1.9]),
                         (E1,  r"$\mathbf{e}$",   BLUE, [-2.0, -2.0, -1.4]),
                         (E2,  r"$\mathbf{e}'$",  BLUE, [2.0, 2.0, -1.4]),
                         (Y1,  r"$\mathbf{x}$",   RED,  [0, 0, 1.6]),
                         (Y2,  r"$\mathbf{x}'$",  RED,  [0, 0, 1.6]),
                         (Xd,  r"$\mathbf{X}$",   "k",  [0, 0, 2.0])):
    ax.scatter(*Q, s=50, facecolors="white", edgecolors=col, linewidths=2,
               depthshade=False, zorder=9)
    ax.text(*(Q + np.array(off)), lab, color=col, fontsize=15)

mid = lambda a, b, t: a + t*(b - a)
for Q, txt in ((Cpd + .10*d + np.array([0, 0, 1.6]),        "baseline"),
               (mid(Cd, Xd, .40)  + np.array([0, 0, -2.0]), "optical ray"),
               (mid(Y2, E2, .50)  + np.array([0, 0, -2.6]), "epipolar line")):
    ax.text(*Q, txt, color="0.35", fontsize=11)

pts = np.vstack([PL1, PL2, [Cd], [Cpd], np.array(list(V3.values())), poly])
lo3, hi3 = pts.min(0), pts.max(0)
ax.set_xlim(lo3[0], hi3[0]); ax.set_ylim(lo3[1], hi3[1]); ax.set_zlim(lo3[2], hi3[2])
ax.set_box_aspect(hi3 - lo3)
ax.view_init(elev=20, azim=-78); ax.set_axis_off()
fig.subplots_adjust(0, 0, 1, 1)
plt.show()


Fix $\mathbf{x}$ in the first image. Its ray lies in the epipolar plane, and so
does everything that follows from it: the world point, wherever along the ray it
happens to be, and therefore its image in the second view. So $\mathbf{x}'$ is
**not free** to vary in the whole image. It has to lie on the epipolar line
where the epipolar plane cuts the second image plane.

We can write that line down without relying $\mathbf{X}$ at all. Two points of the first camera's ray are enough to determine it: the centre
$\mathsf{C}$, and any other point of the ray, for instance
$\mathsf{P}^{+}\mathbf{x}$, where $\mathsf{P}^{+}$ is the pseudo-inverse. That
second point is some point of the ray — the pseudo-inverse returns
some $\mathbf{Y}$ with $\mathsf{P}\mathbf{Y} = \mathbf{x}$. Project both with $\mathsf{P}'$ and join them:

$$\boldsymbol{\ell}' \;=\; (\mathsf{P}'\mathsf{C}) \times
(\mathsf{P}'\mathsf{P}^{+}\mathbf{x}) \;=\; \mathbf{e}' \times
(\mathsf{P}'\mathsf{P}^{+}\mathbf{x}) \;=\;
\underbrace{[\mathbf{e}']_\times \mathsf{P}'\mathsf{P}^{+}}_{\textstyle \mathsf{F}}\;
\mathbf{x}.$$

There is $\mathsf{F}$ again, and this time we have a geometric interpretation: it takes a
point in the first image and returns its **epipolar line** in the second view. We can check that
the correspondences really do lie on the lines it predicts.


In [ ]:
Pplus = np.linalg.pinv(P)
e_p = Pp @ np.append(C, 1.0)                 # the second epipole
F_geo = skew(e_p) @ Pp @ Pplus
F_geo = F_geo / np.linalg.norm(F_geo)

X_h = h(xs); Xp_h = h(xps)
lines = (F_geo @ X_h.T).T                    # one epipolar line per point
dist = np.abs(np.sum(Xp_h * lines, axis=1)) / np.linalg.norm(lines[:, :2], axis=1)

for name, dd in zip(IDS, dist):
    print(f"  {name:3s}  distance of x' from its epipolar line: {dd:.2e} px")


Zero to machine precision, as it must be on exact data.

Now an important property. The map
$\mathbf{x} \mapsto \mathsf{F}\mathbf{x}$ goes from $\mathbb{P}^2$ to
$\mathbb{P}^{2*}$, it transforms points to lines. Moreover, **it is not injective.** Take any two points
of the first image that lie on a common line through the epipole: they belong to
the same epipolar plane, so they must produce the same epipolar line.


In [ ]:
e = np.linalg.svd(F_geo)[2][-1]; e = e / e[2]     # first epipole
x_a = X_h[IDS.index("X9")]
x_b = e + 0.35 * (x_a - e)                        # another point of the same ray

l_a = F_geo @ x_a; l_a /= np.linalg.norm(l_a[:2])
l_b = F_geo @ x_b; l_b /= np.linalg.norm(l_b[:2])

print("x_a :", np.round(x_a[:2], 1))
print("x_b :", np.round(x_b[:2], 1), "  (a different point of the first image)")
print("their epipolar lines differ by:", f"{np.abs(np.abs(l_a) - np.abs(l_b)).max():.2e}")


Two different points in input, the very same line as output. So this map is not invertible, and, since $\mathsf F$ is linear, that means

$$\operatorname{rank} \mathsf{F} = 2, \qquad \det \mathsf{F} = 0 .$$

We could have seen this coming from the construction: $\mathsf{F} =
[\mathbf{e}']_\times \mathsf{P}'\mathsf{P}^{+}$ contains a skew-symmetric
$3\times 3$ factor, and those always have rank two.

The rank deficiency also tells us what the kernel is. $\mathsf{F}\mathbf{e} =
\mathbf{0}$: the epipole is the one point of the first image that has no
epipolar line, because its ray *is* the baseline and its epipolar plane is not
determined. So the epipoles are the null vectors of $\mathsf{F}$.


In [ ]:
def epipoles(F):
    """e spans the right null space of F, e' the right null space of F^T."""
    e  = np.linalg.svd(F)[2][-1];   e  = e / e[2]
    ep = np.linalg.svd(F.T)[2][-1]; ep = ep / ep[2]
    return e, ep

e, ep = epipoles(F_geo)
print("e  =", np.round(e[:2], 0), "   e' =", np.round(ep[:2], 0))
print(f"the frame is {W_IMG} x {H_IMG}, so both fall outside the picture,")
print("to the left: the epipolar lines converge somewhere off-frame.")
print()
print("check: e is the image of the other camera centre")
q = P @ np.append(Cp, 1.0)
print("  P C' =", np.round(q[:2] / q[2], 0))


## Special configurations

Everything so far holds for any pair of cameras. But some kind of relative motions reflect on the structure of $\mathsf F$ itself.

We leave the real images for a moment. We have our 3D model of the origami house, so we can put it in
front of any pair of virtual cameras we like and read off what happens to the fundamental matrix.

In [ ]:
def virtual_pair(direction, baseline=6.0, distance=34.0, f=1800.0,
                 W=1400, H=1050):
    """Two synthetic cameras with the *same* orientation, displaced along one
    axis of the camera frame. The house is the scene; only the motion changes."""
    ctr = X3.mean(axis=0)
    look = ctr + distance * np.array([0.9, -1.0, 0.55]) / np.linalg.norm([0.9, -1.0, 0.55])
    z = ctr - look; z /= np.linalg.norm(z)
    x = np.cross(np.array([0., 0., -1.]), z); x /= np.linalg.norm(x)
    R = np.vstack([x, np.cross(z, x), z])
    K = np.array([[f, 0, W/2], [0, f, H/2], [0, 0, 1.]])
    C1 = look
    C2 = look + baseline * (R.T @ np.asarray(direction, float))
    mk = lambda Cc: K @ np.hstack([R, (-R @ Cc).reshape(3, 1)])
    return mk(C1), mk(C2), W, H


def F_of(Pa, Pb):
    Ca = np.linalg.svd(Pa)[2][-1]; Ca = Ca[:3] / Ca[3]
    F = skew(Pb @ np.append(Ca, 1.0)) @ Pb @ np.linalg.pinv(Pa)
    return F / np.linalg.norm(F)


P_r, Pp_r, W_v, H_v = virtual_pair([1, 0, 0])     # sideways: a rectified pair
P_f, Pp_f, _,   _   = virtual_pair([0, 0, 1])     # straight ahead
F_rect, F_fwd = F_of(P_r, Pp_r), F_of(P_f, Pp_f)

print("a rectified pair — the cameras differ by a translation along their own x axis:")
print(np.array2string(F_rect / np.abs(F_rect).max(), precision=3, suppress_small=True))
print("\nforward motion — the translation is along the optical axis:")
print(np.array2string(F_fwd / np.abs(F_fwd).max(), precision=3, suppress_small=True))
print(f"\nis it skew-symmetric?  |F + F^T| = {np.abs(F_fwd + F_fwd.T).max():.1e}")
e_f, ep_f = epipoles(F_fwd)
print(f"and the two epipoles coincide: e = {e_f[:2].round(1)}, "
      f"e' = {ep_f[:2].round(1)}")

In [ ]:
#| echo: false
#| column: page
#| label: fig-special-configurations
#| fig-cap: >-
#|   Two motions, in synthetic views of the same house. **Left:** the cameras
#|   move sideways and stay parallel, so the epipolar lines are horizontal and
#|   the epipoles have gone to infinity — a match can only have moved along its
#|   own row. **Right:** the cameras move straight ahead, so every epipolar line
#|   passes through one fixed point of the image, the focus of expansion, which
#|   is the same point in both views.
Xh3 = np.c_[X3, np.ones(len(X3))]
proj = lambda Pm: (lambda q: q[:, :2] / q[:, 2:3])(Xh3 @ Pm.T)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), layout="constrained")
for ax, (Pa, Pb, F, ttl) in zip(axes, [
        (P_r, Pp_r, F_rect, "sideways motion: a rectified pair"),
        (P_f, Pp_f, F_fwd,  "forward motion: the epipole stays put")]):
    a, b = proj(Pa), proj(Pb)
    Q = {k: p for k, p in zip(IDS, b)}
    for u, v in EDGES:
        ax.plot(*zip(Q[u], Q[v]), color=GREY, lw=1.4, zorder=3)
    for p in np.c_[a, np.ones(len(a))]:
        seg = clip_line_to_image(F @ p, W_v, H_v)
        if seg is not None:
            ax.plot(seg[:, 0], seg[:, 1], color=ACCENT, lw=1.0, alpha=.9, zorder=2)
    ax.scatter(b[:, 0], b[:, 1], s=34, facecolors="none", edgecolors=BLUE,
               lw=1.6, zorder=5)
    ev = np.linalg.svd(F.T)[2][-1]                  # raw null vector
    finite = abs(ev[2]) > 1e-8 * np.linalg.norm(ev[:2])
    epv = ev / ev[2] if finite else None
    if finite and 0 <= epv[0] < W_v and 0 <= epv[1] < H_v:
        ax.scatter(*epv[:2], s=90, color=BLUE, zorder=6, edgecolors="white", lw=1.4)
        ax.text(epv[0] + 26, epv[1] + 46, r"$\mathbf{e}=\mathbf{e}'$",
                color=BLUE, fontsize=13)
    ax.set_xlim(0, W_v); ax.set_ylim(H_v, 0)
    ax.set_aspect("equal"); ax.set_title(ttl, fontsize=12); ax.axis("off")
plt.show()


**Sideways motion.** With the same orientation and a translation along the camera
$x$ axis, $\mathsf F$ collapses to a single non-zero entry and the constraint
$\mathbf{x}'^\top \mathsf F\mathbf{x} = 0$ reduces to $v = v'$. The epipolar
lines are horizontal, the epipoles have run off to infinity, and a match can only
have moved along its own row. This is a **rectified pair**, and it is what stereo algorithms arrange for on
purpose. The advantage of the configuration is that corresponding points are easy
to look for: given $\mathbf{x}$, you can find $\mathbf{x}'$ by scanning the second
image along the same row. In practice, a two-dimensional search becomes one-dimensional.

**Forward motion.** With a translation along the optical axis, $\mathsf F$ is
**skew-symmetric** (as shown by the  check $\mathsf F + \mathsf F^\top$ above). Skew-symmetry
forces $\mathbf e = \mathbf e'$: the epipole is the same point in both images,
and it is also called the *focus of
expansion*. 

## The pencil of epipolar planes

Everything the fundamental matrix knows is already visible in space, and it fits
in one object: the **pencil of planes through the baseline**. That pencil has a
single degree of freedom (as it turns the planes about the baseline) and every one of its members cuts each image in a line. 

So the epipolar geometry can be also seen as a correspondence between two pencils of lines, and a
correspondence between pencils is a one-dimensional projective map.

Now, consider **a plane of the scene**. A plane induces a
homography $\mathsf H$ between the views. A point of that plane in the first
image $\mathbf x$ determines its corresponding point $\mathbf H \mathbf x$ in the second *exactly*. Take a
point $\mathbf{x}$, cross to the second image through the plane, and join the
result to the epipole. By construction, the join $ [\mathbf e']_\times \mathsf H \mathbf x$ must be the epipolar line $\mathsf F\mathbf{x}$, so

$$\mathsf F \;=\; [\mathbf e']_\times \mathsf H .$$ {#eq-fromH}


The same $\mathsf H$ also carries epipolar lines to epipolar lines, which is the
one-dimensional map itself made explicit:

$$\boldsymbol\ell' \;=\; \mathsf H^{-\top}\boldsymbol\ell .$$ {#eq-linehomography}

One plane is enough to see this. The wall carrying the red door has four vertices
we know, so its homography can be estimated by DLT — and once we have it, we can
check both statements against the geometry we built from the cameras.

The second one is the more telling of the two. Take the epipolar lines of the
*second* view and carry them back to the first, once through $\mathsf F^\top$ and
once through $\mathsf H^\top$. The wall is a small quadrilateral in the corner of
the picture and almost nothing else in the scene lies on it — yet its homography
knows where every epipolar line goes.

In [ ]:
def homography_from_four(a, b):
    """The plane homography carried by four coplanar correspondences (DLT)."""
    A = []
    for (u, v), (up, vp) in zip(a[:, :2], b[:, :2]):
        A += [[-u, -v, -1, 0, 0, 0, up*u, up*v, up],
              [0, 0, 0, -u, -v, -1, vp*u, vp*v, vp]]
    H = np.linalg.svd(np.array(A))[2][-1].reshape(3, 3)
    return H / H[2, 2]


DOOR_WALL = ["X0", "X1", "X5", "X4"]          # the wall the red door is on
idx = [IDS.index(k) for k in DOOR_WALL]
H_door = homography_from_four(X_h[idx], Xp_h[idx])

print("the homography carried by the wall with the door:")
print(np.array2string(H_door, precision=4, suppress_small=True))

# it does send the points of that plane to their partners
q = (H_door @ X_h[idx].T).T
q = q[:, :2] / q[:, 2:3]
print(f"\nit maps the four vertices of the wall onto their partners, "
      f"to {np.abs(q - Xp_h[idx][:, :2]).max():.1e} px")

align = lambda G: np.sign((G * F_geo).sum()) * G / np.linalg.norm(G)
print(f"and [e']x H rebuilds F:                          "
      f"{np.abs(align(skew(ep) @ H_door) - F_geo).max():.1e}")

unit = lambda l: l / np.linalg.norm(l[:2])
gap = max(min(np.abs(unit(H_door.T @ (F_geo @ x)) - unit(F_geo.T @ xp)).max(),
              np.abs(unit(H_door.T @ (F_geo @ x)) + unit(F_geo.T @ xp)).max())
          for x, xp in zip(X_h, Xp_h))
print(f"H^T carries epipolar lines back to epipolar lines: {gap:.1e}")

In [ ]:
#| echo: false
#| column: page
#| label: fig-plane-homography
#| fig-cap: >-
#|   **Left:** four epipolar lines in the second view, $\boldsymbol\ell' =
#|   \mathsf F\mathbf{x}$, one colour each. **Right:** the same four lines
#|   carried back to the first view, drawn twice. In colour, through the
#|   fundamental matrix, $\boldsymbol\ell = \mathsf F^\top\mathbf{x}'$; in white
#|   dashes, through the wall with the door,
#|   $\boldsymbol\ell = \mathsf H^\top\boldsymbol\ell'$. The two agree
#|   everywhere. The blue quadrilateral is the wall whose homography was used;
#|   nothing else in the scene lies on it, and it makes no difference.
import matplotlib.patheffects as pe
HALO = lambda w, c="0.15": [pe.Stroke(linewidth=w, foreground=c), pe.Normal()]

# four vertices whose epipolar lines are as far apart as possible: they all
# pass through the epipole, so what separates them is the angle
ang = np.array([np.arctan2(*(F_geo @ x)[1::-1]) % np.pi for x in X_h])
pick = list(np.argsort(ang)[np.linspace(0, len(ang)-1, 4).astype(int)])
cols = plt.cm.Spectral(np.linspace(.08, .92, len(pick)))

def long_segment(ax, l, W, H, **kw):
    seg = clip_line_to_image(l, W, H)
    if seg is None:
        return
    d = seg[1] - seg[0]; d = d / np.linalg.norm(d)
    far = np.vstack([seg[0] - 2600*d, seg[1] + 2600*d])
    ax.plot(far[:, 0], far[:, 1], **kw)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6), layout="constrained")

axes[0].imshow(Ip)
for k, col in zip(pick, cols):
    long_segment(axes[0], F_geo @ X_h[k], W_IMG, H_IMG,
                 color=col, lw=2.0, zorder=3, path_effects=HALO(3.8))
    axes[0].scatter(*Xp_h[k, :2], s=55, facecolors="none", edgecolors=col,
                    lw=2.2, zorder=6, path_effects=HALO(3.6, "white"))
axes[0].set_title(r"view 2:  $\boldsymbol{\ell}' = \mathsf{F}\mathbf{x}$",
                  fontsize=12)

axes[1].imshow(I)
for k, col in zip(pick, cols):
    long_segment(axes[1], F_geo.T @ Xp_h[k], W_IMG, H_IMG,          # through F
                 color=col, lw=2.0, zorder=3, path_effects=HALO(3.8))
    long_segment(axes[1], H_door.T @ (F_geo @ X_h[k]), W_IMG, H_IMG,  # through H
                 color="white", lw=1.7, ls=(0, (5, 4)), zorder=4,
                 path_effects=HALO(3.5))
    axes[1].scatter(*X_h[k, :2], s=55, facecolors="none", edgecolors=col,
                    lw=2.2, zorder=6, path_effects=HALO(3.6, "white"))
poly = np.vstack([X_h[idx][:, :2], X_h[idx][0, :2]])
axes[1].plot(poly[:, 0], poly[:, 1], color=BLUE, lw=2.4, zorder=5,
             path_effects=HALO(4.2, "white"))
axes[1].set_title(r"view 1:  $\mathsf{F}^\top\mathbf{x}'$  against  "
                  r"$\mathsf{H}^\top\boldsymbol{\ell}'$", fontsize=12)

for ax, pts in zip(axes, (Xp_h, X_h)):
    pad = 300
    ax.set_xlim(pts[:, 0].min() - pad, pts[:, 0].max() + pad)
    ax.set_ylim(pts[:, 1].max() + pad, pts[:, 1].min() - pad)
    ax.axis("off")
plt.show()

The two sets of lines coincide to machine precision. The plane was scaffolding:
any other plane of the scene would have given a different $\mathsf H$ and the
same lines, because $\mathsf F$ is about the cameras and not about what happens
to be in front of them. If the *whole* scene were a single plane the
argument would turn against us: every correspondence would satisfy
$\mathbf{x}' = \mathsf H\mathbf{x}$, which is far stronger than
$\mathbf{x}'^\top\mathsf F\mathbf{x} = 0$, and $[\mathbf e']_\times\mathsf H$
would be a valid answer for *every* choice of $\mathbf e'$. The fundamental
matrix would not be determined at all — a first sighting of the critical
configurations.

## $\mathsf F$ and $\mathsf F^\top$

The construction has treated the two images asymmetrically: we fixed $\mathbf x$
in the first and asked where $\mathbf x'$ could be. Nothing forces that order.
Running the same argument the other way round gives the epipolar line of
$\mathbf x'$ in the first image, and the matrix that does it is the transpose:

$$\boldsymbol\ell' = \mathsf F\,\mathbf x, \qquad
\boldsymbol\ell = \mathsf F^\top\mathbf x' .$$

So a single matrix carries the geometry in both directions, and swapping the
roles of the images means transposing it. Everything comes in pairs accordingly:
$\mathsf F\mathbf e = \mathbf 0$ and $\mathsf F^\top\mathbf e' = \mathbf 0$, the
epipole of one view being the null vector of one matrix and the epipole of the
other the null vector of its transpose.

In [ ]:
#| echo: false
#| column: body
#| label: fig-transpose
#| fig-cap: >-
#|   Transposing $\mathsf F$ swaps the roles of the two images: it carries a point
#|   of the second view to its epipolar line in the first.
L = (F_geo.T @ Xp_h.T).T
pts = pairwise_intersections(L); ctr = np.median(pts, axis=0)
fig, ax = plt.subplots(figsize=(8, 8), layout="constrained")
ax.imshow(I)
for l in L:
    seg = clip_line_to_image(l, W_IMG, H_IMG)
    if seg is None:
        continue
    d = seg[1] - seg[0]; d = d / np.linalg.norm(d)
    far = np.vstack([seg[0] - 2600*d, seg[1] + 2600*d])
    ax.plot(far[:, 0], far[:, 1], color=ACCENT, lw=0.9)
ax.scatter(X_h[:, 0], X_h[:, 1], s=45, facecolors="none",
           edgecolors=BLUE, lw=1.6, zorder=5)
ax.scatter(*ctr, s=60, color=BLUE, zorder=6)
ax.text(ctr[0] + 40, ctr[1] - 40, r"$\mathbf{e}$", color=BLUE, fontsize=13)
ax.set_title(r"$\mathsf{F}^\top$ carries the second view's points"
             "\n" r"to epipolar lines in the first", fontsize=11)
ax.set_xlim(min(0, ctr[0]) - 250, max(W_IMG, ctr[0]) + 250)
ax.set_ylim(max(H_IMG, ctr[1]) + 250, min(700, ctr[1]) - 250)
ax.set_aspect("equal"); ax.axis("off")
plt.show()

## Fundamental matrices and cameras

We have met $\mathsf F$ twice over. **Geometrically**, by following a ray:
$\mathsf F = [\mathbf e']_\times \mathsf P'\mathsf P^{+}$, the epipole joined to
the image of a second point of the ray. **Algebraically**, by expanding
$\det\mathsf L$ from [-@eq-detL], where the bilinear form fell out of the
determinant without any geometry being invoked at all.

The algebraic route says something the geometric one does not. Expanding the
determinant by cofactors, each entry of $\mathsf F$ is a $4\times4$ minor built
from two rows of $\mathsf P$ and two rows of $\mathsf P'$:

$$\mathsf{F}_{ji} \;=\; (-1)^{i+j}\,
\det\!\begin{bmatrix} \widetilde{\mathsf{P}}_i \\ \widetilde{\mathsf{P}}'_j
\end{bmatrix},$$

where $\widetilde{\mathsf{P}}_i$ is $\mathsf P$ with row $i$ removed. So
$\mathsf F$ is computable from the cameras alone, with no correspondences and no
pseudo-inverse — and the rank-two property, which an estimate has to be *forced*
to satisfy, comes out of this construction on its own.

In [ ]:
def F_from_cameras(P, Pp):
    """Each entry of F is a 4x4 minor of the two stacked camera matrices."""
    F = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            rows = ([P[k]  for k in range(3) if k != i] +
                    [Pp[k] for k in range(3) if k != j])
            F[j, i] = (-1)**(i+j) * np.linalg.det(np.vstack(rows))
    return F / np.linalg.norm(F)

F_min = F_from_cameras(P, Pp)
print("rank of F built from the minors:", np.linalg.matrix_rank(F_min),
      "  (nobody asked for it)")
print("agreement with the geometric construction:",
      f"{np.abs(np.sign((F_min*F_geo).sum())*F_min - F_geo).max():.2e}")

# The two expressions agree up to one global scale factor. On corresponding
# pairs both vanish, so we test on pairs that do NOT correspond, where the
# numbers are large and the ratio has something to say.
print("\n   pair          det L        x'^T F x       ratio")
for i, j in [(0, 5), (2, 7), (3, 9), (6, 1)]:
    dL = np.linalg.det(L_matrix(P, Pp, X_h[i], Xp_h[j]))
    q  = Xp_h[j] @ F_min @ X_h[i]
    print(f"   {IDS[i]:>3s}/{IDS[j]:<3s}  {dL:12.4e}  {q:12.4e}  {dL/q:11.4f}")
print("\n   one constant ratio, as a bilinear form defined up to scale must give.")


### And backwards: cameras from $\mathsf{F}$

Both routes go from cameras to $\mathsf F$. Can we go the other way?

Only partly, and the obstruction is worth stating carefully, because it is the
reason uncalibrated reconstruction is *projective* reconstruction.

::: {#thm-cameras-from-f}
## Cameras from the fundamental matrix

A fundamental matrix determines the pair of cameras **up to a projective
transformation of space**. If $\mathsf F$ is the fundamental matrix of
$(\mathsf P, \mathsf P')$, then for every invertible $4\times4$ matrix
$\mathsf{T}$ it is also the fundamental matrix of
$(\mathsf P\mathsf T^{-1}, \mathsf P'\mathsf T^{-1})$; and every pair with that
fundamental matrix is of this form.
:::

The first half is immediate: move every world point to $\mathsf T\mathbf X$ and
replace the cameras by $\mathsf P\mathsf T^{-1}$ and $\mathsf P'\mathsf T^{-1}$.
Nothing in the images changes, since
$\mathsf P\mathsf T^{-1}\mathsf T\mathbf X = \mathsf P\mathbf X$, so $\mathsf F$
cannot possibly tell the two pairs apart. Counting confirms it: two cameras carry
$11 + 11 = 22$ degrees of freedom, a projective transformation of space carries
$15$, and $22 - 15 = 7$ — exactly the freedom of $\mathsf F$. Nothing is left
over, which is the second half.

Within that freedom we may as well make a convenient choice. Sending the first
camera to $[\mathsf I \mid \mathbf 0]$ fixes the world frame to the first
camera's own, and then

$$\mathsf P = [\mathsf I \mid \mathbf 0], \qquad
\mathsf P' = [\,[\mathbf e']_\times \mathsf F \mid \mathbf e'\,]$$

is a pair with the right fundamental matrix — the **canonical pair**. Note the
shape of $\mathsf P'$: its left block is $[\mathbf e']_\times\mathsf F$, which by
[-@eq-fromH] is one of the plane homographies. Choosing a canonical pair is
choosing a plane to call the plane at infinity, and getting that choice wrong is
exactly what makes a projective reconstruction look sheared.

In [ ]:
F_c = F_geo
_, ep_c = epipoles(F_c)

P_can  = np.hstack([np.eye(3), np.zeros((3, 1))])
Pp_can = np.hstack([skew(ep_c) @ F_c, ep_c[:, None]])

F_back = F_from_cameras(P_can, Pp_can)
print("F recovered from the canonical pair, compared with the F we started from:",
      f"{np.abs(np.sign((F_back*F_c).sum())*F_back - F_c).max():.2e}")

# and the cameras it gives are not the ones we started from
print("\nthe canonical first camera is [I | 0], while ours was")
print(np.array2string(P / np.abs(P).max(), precision=3, suppress_small=True))
print("\nsame epipolar geometry, different projective frame.")

### Questions to leave open

**We built $\mathsf F$ from two known cameras. What if we only have the images?**
That is the next notebook, and the answer is less comfortable than it looks: the
constraint is linear in the entries of $\mathsf F$, so eight correspondences give
eight equations — but the matrix they produce is not, in general, of rank two.

**A general $\mathsf F$ has seven degrees of freedom, a skew-symmetric one two,
a rectified one none.** If we knew in advance that the motion was forward, could
we estimate $\mathsf F$ from fewer correspondences? How many?

**The canonical pair chose one plane to call the plane at infinity.** What would
we need to know about the scene, or about the cameras, to make the right choice —
and what does that tell us about which reconstructions are within reach of two
uncalibrated views?

**$\mathsf F$ maps a point to a line, and a line has one degree of freedom left.**
Two views therefore cannot say where along that line the match sits. What extra
information would pin it down?

**The epipoles here fell far outside the frame; under forward motion one sat in
the middle of the picture.** Which motions put the epipole inside, and what does
the epipolar constraint have to say about matches that land near it?

## Further reading

- Longuet-Higgins, H. C. "A computer algorithm for reconstructing a scene from two projections", *Nature* 293, 1981. The original eight-point algorithm.
- Hartley, R. "In defense of the eight-point algorithm", *IEEE TPAMI* 19(6), 1997. Why normalizing the coordinates matters as much as it does.
- Hartley, R. and Zisserman, A. *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004. Chapter 9 for the epipolar geometry, Chapter 11 for the computation of $\mathsf{F}$, and Result 17.1 for the minors.
- Fusiello, A. *Visione Computazionale: tecniche di ricostruzione tridimensionale*, Franco Angeli, 2018. Chapter 5.

---

**Luca Magri** — Computer Vision Dojo
Code MIT · text and figures CC BY-NC-ND 4.0
<https://magrilu.github.io/cv-dojo/>
